In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import (
    ManagedOnlineEndpoint,
    ManagedOnlineDeployment,
)
from azure.identity import DefaultAzureCredential

SUBSCRIPTION_ID = "251b4d15-4de7-4c3d-ad01-8f361090466f"
RESOURCE_GROUP = "rg-mlab1-raheelc2c"
WORKSPACE = "mlw-lab1-raheelc2c"

ml_client = MLClient(
    DefaultAzureCredential(),
    SUBSCRIPTION_ID,
    RESOURCE_GROUP,
    WORKSPACE,
)

print("Connected to Azure ML workspace")


In [ ]:
from azure.ai.ml.entities import ManagedOnlineEndpoint

ENDPOINT_NAME = "knn-maint-raheel"

endpoint = ManagedOnlineEndpoint(
    name=ENDPOINT_NAME,
    description="KNN predictive maintenance notebook model",
    auth_mode="key",
)

ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("Endpoint created:", ENDPOINT_NAME)

In [ ]:
model = ml_client.models.get(
    name="knn-maintenance-notebook",
    label="latest"
)

print("Deploying model:", model.name, "version", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(
    deployment
).result()

endpoint = ml_client.online_endpoints.get(
    name=ENDPOINT_NAME
)
endpoint.traffic = {"blue": 100}

ml_client.online_endpoints.begin_create_or_update(
    endpoint
).result()

print("Deployment ready")

In [ ]:
# Check deployment status
try:
    dep = ml_client.online_deployments.get(
        name="blue",
        endpoint_name=ENDPOINT_NAME
    )
    print("Provisioning state:", dep.provisioning_state)
except Exception as e:
    print("Status check error:", e)

# Get container logs
try:
    logs = ml_client.online_deployments.get_logs(
        name="blue",
        endpoint_name=ENDPOINT_NAME,
        lines=100
    )
    print(logs)
except Exception as e:
    print("Log retrieval error:", e)

In [ ]:
import mlflow
import os

model = ml_client.models.get(
    name="knn-maintenance-notebook",
    version="7"
)

print("Model path:", model.path)
print("Model type:", model.type)
print("Model description:", model.description)

In [ ]:
import mlflow

print("MLflow version:", mlflow.__version__)
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
import mlflow
import os

local_path = mlflow.artifacts.download_artifacts(
    artifact_uri=model.path
)

print("Downloaded model folder:", local_path)
print("Files:", os.listdir(local_path))

In [ ]:
import os
from azure.ai.ml import MLClient

download_dir = "./model_version_7"

ml_client.models.download(
    name="knn-maintenance-notebook",
    version="7",
    download_path=download_dir,
    download_type="custom_model"
)

for root, dirs, files in os.walk(download_dir):
    for filename in files:
        print(os.path.join(root, filename))

In [ ]:
import os

download_dir = "./model_version_7"

ml_client.models.download(
    name="knn-maintenance-notebook",
    version="7",
    download_path=download_dir
)

print("Model files:")
for root, dirs, files in os.walk(download_dir):
    for filename in files:
        print(os.path.join(root, filename))

In [ ]:
for root, dirs, files in os.walk(download_dir):
    for filename in files:
        if filename in ["conda.yaml", "requirements.txt", "MLmodel"]:
            filepath = os.path.join(root, filename)
            print(f"\n--- {filepath} ---")
            with open(filepath, "r", encoding="utf-8") as f:
                print(f.read()[:8000])

In [ ]:
import os

model_dir = "./model_version_7/knn-maintenance-notebook/model_local"

for filename in ["python_env.yaml", "conda.yaml", "requirements.txt"]:
    path = os.path.join(model_dir, filename)
    print(f"\n--- {filename} ---")
    with open(path, "r", encoding="utf-8") as f:
        print(f.read())

print("\n--- MLflow scoring configuration ---")
with open(os.path.join(model_dir, "MLmodel"), "r", encoding="utf-8") as f:
    print(f.read())

In [ ]:
import os

model_dir = "./model_version_7/knn-maintenance-notebook/model_local"

# Add the required packages to requirements.txt
requirements_path = os.path.join(model_dir, "requirements.txt")

with open(requirements_path, "r", encoding="utf-8") as f:
    requirements = f.read()

for package in [
    "azureml-ai-monitoring",
    "azureml-inference-server-http",
]:
    if package not in requirements:
        requirements += f"\n{package}"

with open(requirements_path, "w", encoding="utf-8") as f:
    f.write(requirements.strip() + "\n")


# Add the same packages to conda.yaml
conda_path = os.path.join(model_dir, "conda.yaml")

with open(conda_path, "r", encoding="utf-8") as f:
    conda_text = f.read()

if "azureml-ai-monitoring" not in conda_text:
    conda_text = conda_text.replace(
        "  - scipy==1.15.3",
        "  - scipy==1.15.3\n"
        "  - azureml-ai-monitoring\n"
        "  - azureml-inference-server-http"
    )

with open(conda_path, "w", encoding="utf-8") as f:
    f.write(conda_text)

print("Updated conda.yaml:")
print(open(conda_path, encoding="utf-8").read())

print("\nUpdated requirements.txt:")
print(open(requirements_path, encoding="utf-8").read())

In [ ]:
import os
from azure.ai.ml.entities import Model
from azure.ai.ml.constants import AssetTypes

model_dir = os.path.abspath(
    "./model_version_7/knn-maintenance-notebook/model_local"
)

# Confirm that the corrected files exist
for filename in ["conda.yaml", "requirements.txt", "MLmodel", "model.pkl"]:
    filepath = os.path.join(model_dir, filename)
    print(filename, "exists:", os.path.exists(filepath))

# Register corrected model as a new version
corrected_model = Model(
    name="knn-maintenance-notebook",
    path=model_dir,
    type=AssetTypes.MLFLOW,
    description="KNN model with Azure ML inference dependencies",
)

registered_model = ml_client.models.create_or_update(corrected_model)

print("Registered model:", registered_model.name)
print("New model version:", registered_model.version)

In [ ]:
import os
from azure.ai.ml.entities import Model

model_dir = os.path.abspath(
    "./model_version_7/knn-maintenance-notebook/model_local"
)

# Verify the updated dependency files
for filename in ["conda.yaml", "requirements.txt", "MLmodel", "model.pkl"]:
    filepath = os.path.join(model_dir, filename)
    print(filename, "exists:", os.path.exists(filepath))

# Register corrected model
corrected_model = Model(
    name="knn-maintenance-notebook",
    path=model_dir,
    type="mlflow_model",
    description="KNN model with Azure ML inference dependencies",
)

registered_model = ml_client.models.create_or_update(corrected_model)

print("Registered model:", registered_model.name)
print("New model version:", registered_model.version)

In [ ]:
from azure.ai.ml.entities import ManagedOnlineDeployment

ENDPOINT_NAME = "knn-maint-raheel"

# Get corrected model version 8
model = ml_client.models.get(
    name="knn-maintenance-notebook",
    version="8"
)

print("Deploying corrected model version:", model.version)

deployment = ManagedOnlineDeployment(
    name="blue",
    endpoint_name=ENDPOINT_NAME,
    model=model,
    instance_type="Standard_DS2_v2",
    instance_count=1,
)

ml_client.online_deployments.begin_create_or_update(
    deployment
).result()

print("Deployment created successfully!")

# Route 100% traffic to blue deployment
endpoint = ml_client.online_endpoints.get(
    name=ENDPOINT_NAME
)
endpoint.traffic = {"blue": 100}

ml_client.online_endpoints.begin_create_or_update(
    endpoint
).result()

print("Traffic configured successfully!")

In [ ]:
deployment_status = ml_client.online_deployments.get(
    name="blue",
    endpoint_name="knn-maint-raheel",
)

print("Provisioning state:", deployment_status.provisioning_state)
print("Instance count:", deployment_status.instance_count)

In [ ]:
import json
import os

example_path = (
    "./model_version_7/knn-maintenance-notebook/"
    "model_local/serving_input_example.json"
)

with open(example_path, "r") as file:
    sample_data = json.load(file)

print("Sample input for the deployed model:")
print(json.dumps(sample_data, indent=2))

In [ ]:

import json

sample_path = (
    "./model_version_7/knn-maintenance-notebook/"
    "model_local/serving_input_example.json"
)

with open(sample_path, "r") as file:
    sample_data = json.load(file)

prediction = ml_client.online_endpoints.invoke(
    endpoint_name="knn-maint-raheel",
    deployment_name="blue",
    input_data=sample_data
)

print("Endpoint prediction response:")
print(prediction)


In [ ]:

endpoint = ml_client.online_endpoints.get(
    name="knn-maint-raheel"
)

print("Scoring URI:", endpoint.scoring_uri)

keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-raheel"
)

print("Primary key available:", bool(keys.primary_key))


In [ ]:

import json
import urllib.request
import urllib.error

endpoint = ml_client.online_endpoints.get(
    name="knn-maint-raheel"
)

keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-raheel"
)

sample_path = (
    "./model_version_7/knn-maintenance-notebook/"
    "model_local/serving_input_example.json"
)

with open(sample_path, "r") as file:
    sample_data = json.load(file)

request_body = json.dumps(sample_data).encode("utf-8")

headers = {
    "Content-Type": "application/json",
    "Authorization": "Bearer " + keys.primary_key
}

request = urllib.request.Request(
    endpoint.scoring_uri,
    data=request_body,
    headers=headers,
    method="POST"
)

try:
    with urllib.request.urlopen(request, timeout=120) as response:
        result = response.read().decode("utf-8")
        print("HTTP Status:", response.status)
        print("Prediction response:")
        print(result)

except urllib.error.HTTPError as error:
    print("HTTP Error:", error.code)
    print(error.read().decode("utf-8", "ignore"))

except Exception as error:
    print("Request failed:", str(error))


In [ ]:

# Check the currently deployed model and endpoint configuration

deployment_status = ml_client.online_deployments.get(
    name="blue",
    endpoint_name="knn-maint-raheel"
)

print("Deployment state:", deployment_status.provisioning_state)
print("Deployed model:", deployment_status.model)
print("Scoring URI:", endpoint.scoring_uri)

print("\nDeployment environment:")
print("Environment:", deployment_status.environment)
print("Code configuration:", deployment_status.code_configuration)


In [ ]:

import os

model_dir = (
    "./model_version_7/knn-maintenance-notebook/"
    "model_local"
)

for filename in ["conda.yaml", "requirements.txt"]:
    path = os.path.join(model_dir, filename)
    print(f"\n--- {filename} ---")
    with open(path, "r") as file:
        print(file.read())


In [ ]:

from azure.ai.ml.entities import OnlineDeployment

deployment_logs = ml_client.online_deployments.get_logs(
    name="blue",
    endpoint_name="knn-maint-raheel",
    lines=100
)

print(deployment_logs)


In [ ]:

import json

sample_path = (
    "./model_version_7/knn-maintenance-notebook/"
    "model_local/serving_input_example.json"
)

with open(sample_path, "r") as file:
    sample_data = json.load(file)

print("Top-level keys:", list(sample_data.keys()))
print("Input format:", sample_data.get("dataframe_split", {}).keys())
print("Number of rows:", len(sample_data["dataframe_split"]["data"]))
print("Number of columns:", len(sample_data["dataframe_split"]["columns"]))


In [ ]:

import json
import urllib.request
import urllib.error

endpoint = ml_client.online_endpoints.get(
    name="knn-maint-raheel"
)
keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-raheel"
)

with open(
    "./model_version_7/knn-maintenance-notebook/"
    "model_local/serving_input_example.json",
    "r"
) as file:
    sample_data = json.load(file)

request = urllib.request.Request(
    endpoint.scoring_uri,
    data=json.dumps(sample_data).encode("utf-8"),
    headers={
        "Content-Type": "application/json",
        "Authorization": "Bearer " + keys.primary_key,
    },
    method="POST",
)

try:
    with urllib.request.urlopen(request, timeout=120) as response:
        print("HTTP status:", response.status)
        print(response.read().decode("utf-8"))

except urllib.error.HTTPError as error:
    print("HTTP status:", error.code)
    print("Error response:")
    print(error.read().decode("utf-8", "replace"))


In [ ]:

import json
import urllib.request
import urllib.error

# Correct Azure ML payload format
corrected_payload = {
    "input_data": {
        "columns": sample_data["dataframe_split"]["columns"],
        "data": sample_data["dataframe_split"]["data"],
        "index": [0, 1, 2],
    }
}

request_body = json.dumps(corrected_payload).encode("utf-8")

request = urllib.request.Request(
    endpoint.scoring_uri,
    data=request_body,
    headers={
        "Content-Type": "application/json",
        "Authorization": "Bearer " + keys.primary_key,
    },
    method="POST",
)

try:
    with urllib.request.urlopen(request, timeout=120) as response:
        print("HTTP status:", response.status)
        print("Predictions:", response.read().decode("utf-8"))

except urllib.error.HTTPError as error:
    print("HTTP status:", error.code)
    print("Error response:")
    print(error.read().decode("utf-8", "replace"))


In [ ]:
endpoint = ml_client.online_endpoints.get(
    name="knn-maint-raheel"
)

print("Authentication mode:", endpoint.auth_mode)
print("Scoring URI:", endpoint.scoring_uri)

In [ ]:
import os

keys = ml_client.online_endpoints.get_keys(
    name="knn-maint-raheel"
)

os.environ["AML_ENDPOINT_KEY"] = keys.primary_key
os.environ["AML_ENDPOINT_URL"] = (
    "https://knn-maint-raheel.eastasia.inference.ml.azure.com/score"
)

print("Endpoint URL set:", bool(os.environ["AML_ENDPOINT_URL"]))
print("Primary key set:", bool(os.environ["AML_ENDPOINT_KEY"]))

In [35]:
ml_client.online_endpoints.begin_delete(
    name="knn-maint-raheel"
).result()

print("Endpoint deleted")

.............................................................................Endpoint deleted
